In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from dataclasses import dataclass
from typing import Optional
from kneed import KneeLocator
from scipy.linalg import eig

# Reproducible project paths and output locations

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from inside the DELVE repository.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))



































from project_utils import TABLES_DIR, configure_plots, ensure_output_dirs
from functions import Kernel_matrix, LG_sym, calc_differential_vec, diffusion_map

configure_plots()
ensure_output_dirs()


plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
plt.rcParams["mathtext.it"] = "Times New Roman:italic"
plt.rcParams["mathtext.bf"] = "Times New Roman:bold"


# Ensure imports work whether Jupyter runs from project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
@dataclass
class EntangledSimulation:
    XA: np.ndarray
    XB: np.ndarray

    theta1: np.ndarray
    theta2: np.ndarray

    psiB1: np.ndarray
    psiB2: np.ndarray

    psiA1: Optional[np.ndarray] = None
    psiA2: Optional[np.ndarray] = None


def standardize_columns(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Standardize each observed feature to mean zero and variance one.
    """
    return (X - X.mean(axis=0, keepdims=True)) / (
        X.std(axis=0, keepdims=True) + eps
    )


def simulate_entangled_modalities(
    n: int = 2000,
    n_features_A: int = 12,
    n_features_B: int = 16,
    n_latent_variables: int = 4,
    noise_std: float = 0.02,
    random_state: Optional[int] = 42,
) -> EntangledSimulation:
    """
    Simulate two modalities with nonlinear, entangled latent variables.

    Modality A:
        depends primarily on shared variables theta1 and theta2.

    Modality B:
        depends on theta1, theta2, psiB1, and psiB2.

    If n_latent_variables == 6:
        psiA1 and psiA2 are also generated and may be incorporated into XA.

    Parameters
    ----------
    n : int
        Number of observations.

    n_features_A : int
        Number of observed features in modality A.

    n_features_B : int
        Number of observed features in modality B.

    n_latent_variables : {4, 6}
        Four latents:
            theta1, theta2, psiB1, psiB2

        Six latents:
            theta1, theta2, psiA1, psiA2, psiB1, psiB2

    noise_std : float
        Standard deviation of additive isotropic Gaussian noise.

    random_state : int or None
        Random seed.

    Returns
    -------
    EntangledSimulation
        Observations and their underlying latent variables.
    """
    if n_latent_variables not in (4, 6):
        raise ValueError("n_latent_variables must be either 4 or 6.")

    if n_features_A < 1 or n_features_B < 1:
        raise ValueError("The number of observed features must be positive.")

    rng = np.random.default_rng(random_state)

    # -----------------------------------------------------
    # 1. Latent variables
    # -----------------------------------------------------
    theta1 = 0.5*rng.standard_normal(n)+0.32
    theta2 = 0.3*rng.standard_normal(n)

    psiB1 = 1.2*rng.standard_normal(n)+3
    psiB2 = 1.9*rng.standard_normal(n)+1.5

    psiA1 = None
    psiA2 = None

    if n_latent_variables == 6:
        psiA1 = rng.standard_normal(n)
        psiA2 = rng.standard_normal(n)

    # -----------------------------------------------------
    # 2. Nonlinear feature dictionary for modality A
    # -----------------------------------------------------
    # Every feature depends on theta1, theta2, or both.
    # Therefore, the observed coordinates are not aligned
    # with the individual latent factors.
    features_A = [
        theta1**2 + 0.35 * theta2**2,
        np.sin(theta1 + 0.50 * theta2),
        np.cos(0.75 * theta1 - theta2),
        theta1**3 + 0.35 * theta2**3,
        np.tanh(theta1 - 0.60 * theta2),
        theta1**2 + 0.30 * theta2,
        theta2**2 - 0.25 * theta1,
        np.sin(theta1 + theta2),
        np.exp(-0.25 * (theta1**2 + theta2**2)),
        np.sign(theta1 + theta2) * np.sqrt(np.abs(theta1 + theta2)),
        np.sin(theta1) + np.cos(theta2),
        (theta1 + theta2) / (1.0 + np.abs(theta1 - theta2)),
    ]

    # In the six-latent-variable setting, XA also contains
    # its own modality-specific variables.
    if n_latent_variables == 6:
        features_A.extend([
            np.sin(theta1 + 0.60 * psiA1),
            np.cos(theta2 - 0.50 * psiA2),
            theta1 * psiA1 + 0.30 * theta2,
            theta2 * psiA2 - 0.25 * theta1,
            np.tanh(psiA1 + psiA2 + 0.30 * theta1),
            np.sin(psiA1 * psiA2 + theta2),
        ])

    # -----------------------------------------------------
    # 3. Nonlinear feature dictionary for modality B
    # -----------------------------------------------------
    # The distinctive variables psiB1 and psiB2 are mixed
    # together with the shared variables theta1 and theta2.
    features_B = [
        theta1**2 + 0.30 * theta2 + 0.20,
        theta2**2 - 0.25 * theta1 + 0.20,

        0.50 * psiB1**2,
        theta2**2 - 0.60 * psiB2,
        
        0.30 * psiB2**2,
        theta1**2 - 0.20 * psiB1,
        

        np.sin(theta1 + theta2 + 0.40 ),
        np.cos(theta1 - theta2 + 0.40),

        theta1 **3 + psiB1,
        theta2 **3 + psiB2,

        theta1 * theta2 + 0.40 * psiB1,
        psiB1 + psiB2**5 + 0.25 * theta1,

        np.sin(theta1 + psiB1 - 0.30 * psiB2),
        np.sin(theta2 + psiB2 + 0.30 * psiB1),

        psiB1**2 + 0.30 * theta1,
        psiB2**2 - 0.30 * theta2,

        np.sin(theta1 + psiB1 + theta2 + psiB2),
        np.exp(
            -0.20 * (
                theta1**2
                + theta2**2
                + psiB1
                + psiB2
            )
        ),

        np.sin(theta1 + psiB1) * np.cos(theta2 + psiB2),
        (theta1 + psiB1) / (
            1.0 + np.abs(theta2 - psiB2)
        ),
    ]

    # -----------------------------------------------------
    # 4. Add random nonlinear mixtures if more observed
    #    features are requested than listed above
    # -----------------------------------------------------
    def generate_random_features(
        latent_matrix: np.ndarray,
        n_extra: int,
    ) -> list[np.ndarray]:
        extra_features = []

        for _ in range(n_extra):
            weights1 = rng.normal(size=latent_matrix.shape[1])
            weights2 = rng.normal(size=latent_matrix.shape[1])

            weights1 /= np.linalg.norm(weights1) + 1e-12
            weights2 /= np.linalg.norm(weights2) + 1e-12

            z1 = latent_matrix @ weights1
            z2 = latent_matrix @ weights2

            feature_type = rng.integers(0, 5)

            if feature_type == 0:
                feature = np.sin(z1) + 0.30 * z2
            elif feature_type == 1:
                feature = np.cos(z1) * np.tanh(z2)
            elif feature_type == 2:
                feature = z1 * z2
            elif feature_type == 3:
                feature = np.tanh(z1 + 0.50 * z2)
            else:
                feature = z1**2 + 0.25 * z2

            extra_features.append(feature)

        return extra_features

    if n_latent_variables == 4:
        latents_A = np.column_stack([theta1, theta2])
    else:
        latents_A = np.column_stack([
            theta1,
            theta2,
            psiA1,
            psiA2,
        ])

    latents_B = np.column_stack([
        theta1,
        theta2,
        psiB1,
        psiB2,
    ])

    if n_features_A > len(features_A):
        features_A.extend(
            generate_random_features(
                latents_A,
                n_features_A - len(features_A),
            )
        )

    if n_features_B > len(features_B):
        features_B.extend(
            generate_random_features(
                latents_B,
                n_features_B - len(features_B),
            )
        )

    # Keep only the requested number of observed features.
    XA = np.column_stack(features_A[:n_features_A])
    XB = np.column_stack(features_B[:n_features_B])

    # -----------------------------------------------------
    # 5. Standardize clean signal, then add Gaussian noise
    # -----------------------------------------------------
    XA = standardize_columns(XA)
    XB = standardize_columns(XB)

    XA += noise_std * rng.standard_normal(XA.shape)
    XB += noise_std * rng.standard_normal(XB.shape)

    return EntangledSimulation(
        XA=XA,
        XB=XB,
        theta1=theta1,
        theta2=theta2,
        psiA1=psiA1,
        psiA2=psiA2,
        psiB1=psiB1,
        psiB2=psiB2,
    )
    

In [ ]:
sim = simulate_entangled_modalities(
    n=2000,
    n_features_A=12,
    n_features_B=16,
    n_latent_variables=4,
    noise_std=0.02,
    random_state=42,
)

XA = sim.XA
XB = sim.XB

theta1 = sim.theta1
theta2 = sim.theta2
psiB1 = sim.psiB1
psiB2 = sim.psiB2

print("XA shape:", XA.shape)
print("XB shape:", XB.shape)

In [ ]:
# Core operators/eigenvectors for all methods on the representative sample
P1, Q1, K1 = diffusion_map(XA, adaptive=1600)
P2, Q2, K2 = diffusion_map(XB, adaptive=1600)

L1, d1, v1 = LG_sym(K1)
L2, d2, v2 = LG_sym(K2)

kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.25)
tau1 = kl.knee
kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.25)
tau2 = kl.knee
print(tau1)
print(tau2)

In [ ]:
_, u1 = calc_differential_vec(L2, v1, tau1)
_, u2 = calc_differential_vec(L1, v2, tau2)

In [ ]:
distinctive_targets = np.column_stack([
    psiB1,
    psiB2,
    psiB1 + psiB2,
    psiB1 - psiB2,
    psiB1 * psiB2,
    psiB1**2,
    psiB2**2,
    psiB1**2 + psiB2**2,
    np.sin(psiB1),
    np.sin(psiB2),
])


corrs = [
    abs(spearmanr(u1[:,0], distinctive_targets[:, j]).statistic)
    for j in range(distinctive_targets.shape[1])
]

best_corr = max(corrs)
best_target = np.argmax(corrs)

In [ ]:
corrs

In [ ]:
L_shared = L1@L2 + L2@L1

ds, v_shared = np.linalg.eigh(L_shared)
idx_s = np.argsort(ds)[::-1]
v_shared = v_shared[:,idx_s]

In [ ]:
V_1 = np.concatenate((v_shared[:,:5], u1[:,0:1]), axis = 1)
K_V1 = Kernel_matrix(V_1,1650, False)

L_V1, d_VA, v_V1 = LG_sym(K_V1)

# tau_V1, u_V1, scores_V1 = choose_tau(L_own=L2,L_other=L_V1,v_other=v_V1)

kl = KneeLocator(np.arange(len(d_VA)), d_VA, curve="convex", direction="decreasing",S=0.5)
tauV1 = kl.knee
print(tauV1)

L2_nr, s_nr, u1_nr = calc_differential_vec(L2,v_V1,tauV1,"yes")

In [ ]:
corrs = [
    abs(spearmanr(u1_nr[:,0], distinctive_targets[:, j]).statistic)
    for j in range(distinctive_targets.shape[1])
]

corrs

## Repeated simulation

The following cells repeat the same simulation and report mean (standard deviation) recovery of $\psi_{B1}$ and $\psi_{B2}$. Only the leading vectors of Algorithm 1 and Algorithm 2 are used.


In [ ]:
# Candidate functions used only for the secondary correlation table.
target_names = [
    r"$\psi_{B1}$",
    r"$\psi_{B2}$",
    r"$\psi_{B1}+\psi_{B2}$",
    r"$\psi_{B1}-\psi_{B2}$",
    r"$\psi_{B1}\psi_{B2}$",
    r"$\psi_{B1}^2$",
    r"$\psi_{B2}^2$",
    r"$\psi_{B1}^2+\psi_{B2}^2$",
    r"$\sin(\psi_{B1})$",
    r"$\sin(\psi_{B2})$",
]


def run_once(seed):
    sim = simulate_entangled_modalities(
        n=2000,
        n_features_A=12,
        n_features_B=16,
        n_latent_variables=4,
        noise_std=0.02,
        random_state=seed,
    )

    XA, XB = sim.XA, sim.XB
    psiB1, psiB2 = sim.psiB1, sim.psiB2

    P1, Q1, K1 = diffusion_map(XA, adaptive=1600)
    P2, Q2, K2 = diffusion_map(XB, adaptive=1600)
    L1, d1, v1 = LG_sym(K1)
    L2, d2, v2 = LG_sym(K2)

    tau1 = KneeLocator(
        np.arange(len(d1)), d1,
        curve="convex", direction="decreasing", S=0.25
    ).knee

    # Algorithm 1: leading differential vector for modality B.
    _, u1 = calc_differential_vec(L2, v1, tau1)

    # Algorithm 2: remove the remaining shared information.
    _, v_shared = np.linalg.eigh(L1 @ L2 + L2 @ L1)
    v_shared = v_shared[:, ::-1]

    V1 = np.c_[v_shared[:, :5], u1[:, 0]]
    L_V1, d_V1, v_V1 = LG_sym(Kernel_matrix(V1, 1650, False))
    tauV1 = KneeLocator(
        np.arange(len(d_V1)), d_V1,
        curve="convex", direction="decreasing", S=0.5
    ).knee
    _, _, u1_nr = calc_differential_vec(L2, v_V1, tauV1, "yes")

    # FKT: leading modality-B vector.
    g1 = np.diag(np.sum(K1, axis=0)) - K1
    g2 = np.diag(np.sum(K2, axis=0)) - K2
    m1 = g1 + 1e-6 * np.eye(g1.shape[0])
    m2 = g2 + 1e-6 * np.eye(g2.shape[0])
    fk1 = np.linalg.solve(m1 + m2, m1)
    fk_values, fk_vectors = eig(fk1)
    fkt_B = np.real(fk_vectors[:, np.argsort(np.real(fk_values))[::-1][0]])

    # Shnitzer et al.: leading real and imaginary differential vectors.
    D = P2 @ Q1 - P1 @ Q2
    sh_values, sh_vectors = eig(D)
    sh_real = np.real(sh_vectors[:, np.argsort(np.real(sh_values))[::-1][0]])
    sh_imag = np.imag(sh_vectors[:, np.argsort(np.imag(sh_values))[::-1][0]])

    targets = np.column_stack([
        psiB1,
        psiB2,
        psiB1 + psiB2,
        psiB1 - psiB2,
        psiB1 * psiB2,
        psiB1**2,
        psiB2**2,
        psiB1**2 + psiB2**2,
        np.sin(psiB1),
        np.sin(psiB2),
    ])

    # Main table: direct recovery of psiB1 and psiB2.
    main = [
        abs(spearmanr(u1[:, 0], psiB1).statistic),
        abs(spearmanr(u1[:, 0], psiB2).statistic),
        abs(spearmanr(u1_nr[:, 0], psiB1).statistic),
        abs(spearmanr(u1_nr[:, 0], psiB2).statistic),
        abs(spearmanr(fkt_B, psiB1).statistic),
        abs(spearmanr(fkt_B, psiB2).statistic),
        abs(spearmanr(sh_real, psiB1).statistic),
        abs(spearmanr(sh_real, psiB2).statistic),
        abs(spearmanr(sh_imag, psiB1).statistic),
        abs(spearmanr(sh_imag, psiB2).statistic),
    ]

    # Secondary table: correlations of the two DELVE vectors with all targets.
    corr_alg1 = [abs(spearmanr(u1[:, 0], target).statistic) for target in targets.T]
    corr_alg2 = [abs(spearmanr(u1_nr[:, 0], target).statistic) for target in targets.T]

    return main + corr_alg1 + corr_alg2

In [ ]:
B = 100
results = np.array([run_once(seed) for seed in range(B)])

In [ ]:
# ---------------------------------------------------------
# Main recovery table
# ---------------------------------------------------------
summary = pd.DataFrame(
    index=[r"$\psi_{B1}$", r"$\psi_{B2}$"]
)

summary["DELVE: Algorithm 1"] = [
    f"{results[:, 0].mean():.3f} ({results[:, 0].std():.3f})",
    f"{results[:, 1].mean():.3f} ({results[:, 1].std():.3f})",
]
summary["DELVE: Algorithm 2"] = [
    f"{results[:, 2].mean():.3f} ({results[:, 2].std():.3f})",
    f"{results[:, 3].mean():.3f} ({results[:, 3].std():.3f})",
]
summary["FKT"] = [
    f"{results[:, 4].mean():.3f} ({results[:, 4].std():.3f})",
    f"{results[:, 5].mean():.3f} ({results[:, 5].std():.3f})",
]
summary["Shnitzer (real)"] = [
    f"{results[:, 6].mean():.3f} ({results[:, 6].std():.3f})",
    f"{results[:, 7].mean():.3f} ({results[:, 7].std():.3f})",
]
summary["Shnitzer (imag)"] = [
    f"{results[:, 8].mean():.3f} ({results[:, 8].std():.3f})",
    f"{results[:, 9].mean():.3f} ({results[:, 9].std():.3f})",
]

display(summary)
summary.to_latex(TABLES_DIR / "entangled_recovery_summary.tex", escape=False)

# ---------------------------------------------------------
# Secondary table: strongest other functions of psiB1, psiB2
# ---------------------------------------------------------
n_targets = len(target_names)
alg1_corrs = results[:, 10:10 + n_targets]
alg2_corrs = results[:, 10 + n_targets:10 + 2*n_targets]

# Exclude psiB1 and psiB2 because they are already in the main table.
other_idx = np.arange(2, n_targets)
top1 = other_idx[np.argsort(alg1_corrs[:, other_idx].mean(axis=0))[::-1][:3]]
top2 = other_idx[np.argsort(alg2_corrs[:, other_idx].mean(axis=0))[::-1][:3]]

top_correlations = pd.DataFrame(index=["1", "2", "3"])
top_correlations["Algorithm 1: function"] = [target_names[i] for i in top1]
top_correlations["Algorithm 1: mean (std)"] = [
    f"{alg1_corrs[:, i].mean():.3f} ({alg1_corrs[:, i].std():.3f})"
    for i in top1
]
top_correlations["Algorithm 2: function"] = [target_names[i] for i in top2]
top_correlations["Algorithm 2: mean (std)"] = [
    f"{alg2_corrs[:, i].mean():.3f} ({alg2_corrs[:, i].std():.3f})"
    for i in top2
]

display(top_correlations)
top_correlations.to_latex(
    TABLES_DIR / "entangled_top_correlations.tex",
    escape=False,
)